In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io.arff import loadarff
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dropout, Dense, LSTM

In [ ]:
fp= '../ScenarioA2/TimeBasedFeatures-Dataset-15s-VPN.arff'
raw_data = loadarff(fp)

In [ ]:
df_A2 = pd.DataFrame(raw_data[0])
df_A2

In [ ]:
df=df_A2
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['duration']=le.fit_transform(df.duration)
df['total_fiat']=le.fit_transform(df.total_fiat)
df['total_biat']=le.fit_transform(df.total_biat)
df['min_fiat']=le.fit_transform(df.min_fiat)
df['max_fiat']=le.fit_transform(df.max_fiat)
df['max_biat']=le.fit_transform(df.max_biat)
df['mean_fiat']=le.fit_transform(df.mean_fiat)
df['mean_biat']=le.fit_transform(df.mean_biat)
df['flowPktsPerSecond']=le.fit_transform(df.flowPktsPerSecond)
df['flowBytesPerSecond']=le.fit_transform(df.flowBytesPerSecond)
df['min_flowiat']=le.fit_transform(df.min_flowiat)
df['max_flowiat']=le.fit_transform(df.max_flowiat)
df['mean_flowiat']=le.fit_transform(df.mean_flowiat)
df['std_flowiat']=le.fit_transform(df.std_flowiat)
df['min_active']=le.fit_transform(df.min_active)
df['mean_active']=le.fit_transform(df.mean_active)
df['max_active']=le.fit_transform(df.max_active)
df['std_active']=le.fit_transform(df.std_active)
df['min_idle']=le.fit_transform(df.min_idle)
df['mean_idle']=le.fit_transform(df.mean_idle)
df['max_idle']=le.fit_transform(df.max_idle)
df['std_idle']=le.fit_transform(df.std_idle)
df_A2=df

In [ ]:
df_A2['class1'].value_counts()

In [ ]:
df_A2['class1'] = df_A2['class1'].replace([b'VPN-BROWSING'],1).replace([b'VPN-VOIP'],2).replace([b'VPN-FT'],3).replace(
[b'VPN-CHAT'],4).replace([b'VPN-P2P'],5).replace([b'VPN-MAIL'],6).replace([b'VPN-STREAMING'],7)

In [ ]:
X = df_A2.drop(columns=["class1"])
y = df_A2["class1"]
random_state = 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=random_state, stratify=y)

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.fit_transform(X_test)
X_train3D = X_train.reshape(X_train.shape[0], 23, 1).astype('float32')
X_test3D = X_test.reshape(X_test.shape[0], 23, 1).astype('float32')

In [ ]:
modelCNN = Sequential()
modelCNN.add(Conv1D(filters=32, kernel_size=5, input_shape=(23, 1)))
modelCNN.add(MaxPooling1D(pool_size=5 ))
modelCNN.add(Flatten())
modelCNN.add(Dense(1000, activation='relu'))
modelCNN.add(Dense(8, activation='softmax'))

In [ ]:
modelRNN = Sequential()
modelRNN.add(LSTM(128,  activation='relu', input_shape=(23,1), return_sequences=True))
modelRNN.add(Dropout(0.2))
modelRNN.add(LSTM(64, activation='relu'))
modelRNN.add(Dropout(0.2))
modelRNN.add(Dense(32, activation='relu'))
modelRNN.add(Dropout(0.2))
modelRNN.add(Dense(8, activation='softmax'))

In [ ]:
modelDNN = Sequential()
modelDNN.add(Dense(128, activation="relu", input_dim=23))
modelDNN.add(Dense(64, activation='relu'))
modelDNN.add(Dense(32, activation='relu'))
modelDNN.add(Dense(16, activation='relu'))
modelDNN.add(Dense(8, activation='softmax'))

In [ ]:
modelCNN.summary()

In [ ]:
modelRNN.summary()

In [ ]:
modelDNN.summary()

In [ ]:
modelCNN.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
modelCNN.fit(X_train3D, y_train, epochs=100)

In [ ]:
modelRNN.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
modelRNN.fit(X_train3D, y_train, epochs = 30)

In [ ]:
modelDNN.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
modelDNN.fit(X_train, y_train, epochs=30, batch_size = 10, validation_data=(X_test, y_test))

In [ ]:
CNN_y_pred = modelCNN.predict(X_test3D)   
RNN_y_pred = modelRNN.predict(X_test3D)
DNN_y_pred = modelDNN.predict(X_test)

In [ ]:
CNN_y_pred = np.argmax(CNN_y_pred, axis=1)
RNN_y_pred = np.argmax(RNN_y_pred, axis=1)
DNN_y_pred = np.argmax(DNN_y_pred, axis=1)

In [ ]:
CNNResult = accuracy_score(y_test, CNN_y_pred)
RNNResult = accuracy_score(y_test, RNN_y_pred)
DNNResult = accuracy_score(y_test, DNN_y_pred)

In [ ]:
print('Scenario A2 CNN Accuracy Score: {0:0.4f}'.format(CNNResult))
print('Scenario A2 RNN Accuracy Score: {0:0.4f}'.format(RNNResult))
print('Scenario A2 DNN Accuracy Score: {0:0.4f}'.format(DNNResult))

In [ ]:
CNN_cm = confusion_matrix(y_test, CNN_y_pred)
RNN_cm = confusion_matrix(y_test, RNN_y_pred)
DNN_cm = confusion_matrix(y_test, DNN_y_pred)

In [ ]:
target_names = ["b'VOIP'","b'BROWSING'","b'FT'","b'P2P'","b'CHAT'","b'STREAMING'","b'MAIL'"] 
CNN_cm_df = pd.DataFrame(CNN_cm,
                     index = target_names, 
                     columns = target_names)
RNN_cm_df = pd.DataFrame(RNN_cm,
                     index = target_names, 
                     columns = target_names)
DNN_cm_df = pd.DataFrame(DNN_cm,
                     index = target_names, 
                     columns = target_names)

In [ ]:
def createConfusionMatrix(matrix,title):
    plt.figure(figsize=(10,10))
    sns.heatmap(matrix, annot=True, fmt='g')
    plt.title(title, fontsize=20)
    plt.ylabel('Actual Values')
    plt.xlabel('Predicted Values')
    plt.show()

In [ ]:
CNN_title = "Scenario A2 CNN Model Confusion Matrix"
createConfusionMatrix(CNN_cm_df, CNN_title)

In [ ]:
RNN_title = "Scenario A2 RNN Model Confusion Matrix"
createConfusionMatrix(RNN_cm_df, RNN_title)

In [ ]:
DNN_title = "Scenario A2 DNN Model Confusion Matrix"
createConfusionMatrix(DNN_cm_df, DNN_title)

In [ ]:
CNN_report = classification_report(y_test, CNN_y_pred, target_names=target_names)
RNN_report = classification_report(y_test, RNN_y_pred, target_names=target_names)
DNN_report = classification_report(y_test, DNN_y_pred, target_names=target_names)

In [ ]:
print("     Scenario A2 CNN Classification Report\n", CNN_report)

In [ ]:
print("     Scenario A2 RNN Classification Report\n", RNN_report)

In [ ]:
print("     Scenario A2 DNN Classification Report\n", DNN_report)